# S1-03: Temperature와 스트리밍
**Skilljar L10-L11: Temperature / Response Streaming**

## 학습 목표
- Temperature 파라미터의 동작 원리와 용도별 설정을 이해한다
- Temperature 0.0 (결정적) vs 1.0 (다양) 응답을 비교한다
- `stream=True`로 기본 스트리밍을 구현한다
- `client.messages.stream()` + `text_stream`으로 간편 스트리밍을 구현한다
- `get_final_message()`로 스트리밍 후 메타데이터를 활용한다

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 클라이언트 생성 및 헬퍼 함수
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages: list, text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list, text: str):
    messages.append({"role": "assistant", "content": text})

def chat(messages: list, system: str = None, temperature: float = 1.0) -> str:
    """Claude API 호출 (temperature 포함)"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

print("설정 완료")

## 1. Temperature란?

Temperature는 토큰 확률 분포의 **뾰족함(sharpness)**을 조절한다:

| Temperature | 효과 | 건축공학 비유 |
|---|---|---|
| **0.0** | 최고 확률 토큰을 항상 선택 (결정적) | 설계기준서 — 정해진 답만 |
| **0.0 ~ 0.3** | 상위 소수 토큰에 집중 | 구조계산 — 검증된 방법만 |
| **0.4 ~ 0.7** | 적당한 다양성 | 설계 대안 검토 — 합리적 범위 내 |
| **0.8 ~ 1.0** | 확률이 고르게 분산 | 초기 아이디어 스케치 — 자유롭게 |

> Anthropic API의 temperature 기본값은 **1.0**이다. 공학 작업에서는 반드시 낮은 값을 지정해야 한다.

In [ ]:
# Temperature 비교 실험 1: 같은 질문, 다른 temperature

question = "콘크리트 구조물의 내구성을 높이는 방법을 3가지 제시해주세요."

# Temperature 0.0 — 결정적 (매번 동일한 답변)
msgs_low = []
add_user_message(msgs_low, question)
result_low = chat(msgs_low, temperature=0.0)
print("=== Temperature 0.0 (결정적) ===")
print(result_low)

In [ ]:
# Temperature 1.0 — 다양한 응답 (매번 다른 답변)
msgs_high = []
add_user_message(msgs_high, question)
result_high = chat(msgs_high, temperature=1.0)
print("=== Temperature 1.0 (다양) ===")
print(result_high)

In [ ]:
# Temperature 0.0은 매번 동일한 결과를 보장한다 — 3번 호출로 확인
print("=== Temperature 0.0 을 3번 호출 ===")
for i in range(3):
    msgs = []
    add_user_message(msgs, "한국에서 가장 높은 건물의 이름과 높이를 한 문장으로 답해줘.")
    result = chat(msgs, temperature=0.0)
    print(f"  [{i+1}] {result}")

In [ ]:
# 건축공학 적용: 구조 계산 vs 브레인스토밍

# 구조 계산 — 정확성 우선 (temperature = 0.0)
msgs_calc = []
add_user_message(msgs_calc,
    "500x500 RC 기둥, fck=24MPa, fy=400MPa일 때 "
    "축 하중 강도 Pn을 KDS 기준으로 계산하라."
)
result_precise = chat(msgs_calc, temperature=0.0)
print("=== Temperature 0.0 (구조 계산) ===")
print(result_precise)

print("\n" + "="*60 + "\n")

# 설계 아이디어 — 창의성 우선 (temperature = 0.9)
msgs_idea = []
add_user_message(msgs_idea,
    "20층 주거 건물의 횡력저항시스템으로 "
    "가능한 구조 대안을 브레인스토밍하라."
)
result_creative = chat(msgs_idea, temperature=0.9)
print("=== Temperature 0.9 (브레인스토밍) ===")
print(result_creative)

## 2. 스트리밍 (Response Streaming)

일반 API 호출은 전체 응답이 생성될 때까지 기다린 후 한 번에 반환한다.

스트리밍은 응답을 **토큰 단위로 실시간** 전달하여 사용자 경험을 개선한다.

In [ ]:
# 기본 스트리밍 — stream=True (Raw Events)
# 각 이벤트 객체를 직접 확인

stream = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[{"role": "user", "content": "RC 보의 전단설계 3단계를 간략히 설명하라."}],
    stream=True
)

# 이벤트 종류를 확인
for event in stream:
    print(f"[{event.type}]", end=" ")
    if hasattr(event, 'delta') and hasattr(event.delta, 'text'):
        print(event.delta.text, end="")
    else:
        print()

### 스트림 이벤트 구조

| 이벤트 | 설명 |
|---|---|
| `message_start` | 메시지 시작 |
| `content_block_start` | 콘텐츠 블록 시작 |
| `content_block_delta` | **텍스트 조각 전달** (반복) |
| `content_block_stop` | 콘텐츠 블록 종료 |
| `message_delta` | 메시지 메타데이터 |
| `message_stop` | 메시지 완전 종료 |

In [ ]:
# 간편 스트리밍 — client.messages.stream() + text_stream
# 대부분의 경우 이 방법이 편리하다

print("=== 실시간 스트리밍 출력 ===")

with client.messages.stream(
    model=model,
    max_tokens=500,
    messages=[{"role": "user", "content": "RC 보의 처짐 검토 절차를 3단계로 설명하라."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)  # flush=True: 버퍼링 없이 즉시 출력

print()  # 줄바꿈

In [ ]:
# get_final_message() — 스트리밍 후 메타데이터 활용

print("=== 스트리밍 + 메타데이터 ===")

with client.messages.stream(
    model=model,
    max_tokens=500,
    messages=[{"role": "user", "content": "콘크리트 균열 종류 3가지를 설명하라."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

    # 스트림 종료 후 최종 메시지 획득
    final_message = stream.get_final_message()

print(f"\n\n--- 메타데이터 ---")
print(f"모델: {final_message.model}")
print(f"종료 사유: {final_message.stop_reason}")
print(f"입력 토큰: {final_message.usage.input_tokens}")
print(f"출력 토큰: {final_message.usage.output_tokens}")

---
## 건축공학 실습 과제

### 과제: 구조계산서 실시간 생성

아래 조건으로 구조계산서를 **스트리밍**으로 실시간 생성하세요.

**요구사항:**
1. 시스템 프롬프트: 구조공학 전문가 역할
2. Temperature: 0.0 (정확성 우선)
3. 스트리밍: `client.messages.stream()` + `text_stream` 사용
4. 스트리밍 완료 후 `get_final_message()`로 토큰 사용량 출력
5. 검토 대상: RC 보의 휨 설계 검토
   - 보 단면: 300mm x 600mm
   - fck = 27 MPa, fy = 400 MPa
   - 인장철근: 4-D25
   - 설계 휨모멘트 (Mu): 250 kN-m

In [ ]:
# TODO: 구조계산서 실시간 스트리밍 생성

system_structural = """당신은 한국 건축구조 설계기준(KDS)에 정통한 구조공학 전문가입니다.
계산 과정을 단계별로 상세히 보여주세요.
적용 KDS 조항 번호를 명시하세요.
결과를 표 형식으로 정리하세요."""

prompt = """다음 RC 보의 휨 설계 적정성을 검토해주세요.

설계 조건:
- 보 단면: 300mm x 600mm
- 콘크리트 강도 (fck): 27 MPa
- 철근 항복강도 (fy): 400 MPa
- 인장철근: 4-D25
- 설계 휨모멘트 (Mu): 250 kN-m

검토 항목:
1. 공칭 휨강도 (Mn) 계산
2. 설계 휨강도 (phi*Mn) 산정
3. Mu <= phi*Mn 여부 판정
4. 최소/최대 철근비 검토"""

print("=== RC 보 휨 설계 검토 (실시간 스트리밍) ===")
print()

with client.messages.stream(
    model=model,
    max_tokens=2000,
    system=system_structural,
    temperature=0.0,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

    final = stream.get_final_message()

print(f"\n\n--- 토큰 사용량 ---")
print(f"입력: {final.usage.input_tokens} 토큰")
print(f"출력: {final.usage.output_tokens} 토큰")
print(f"종료 사유: {final.stop_reason}")